# MBR decoding on the seed-42 checkpoint

## ⚠️ BEFORE RUNNING
1. **Settings → Accelerator → `GPU T4 x2`** (P100 is sm_60 and cannot run this)
2. **Settings → Internet → On**
3. The training notebook `nascenia-banglat5-v2` must be attached as a **Notebook input**
   with its checkpoint saved — do **File → Save Version → Quick Save** on it first.

## Why this matters more than retraining

TRAIN-01 finished at **Token F1 0.2051** — statistically indistinguishable from TF-IDF
retrieval (0.2047) and *below* a constant string (0.2669). Both the model and retrieval
emit **specific** content; the constant emits **generic** content, and specific-but-wrong
loses more precision than it gains in recall.

**MBR attacks exactly that.** It samples N candidates and picks the one closest to their
consensus — structurally biased toward the generic centre, which is where the points are.

Retraining costs ~7 h. This costs ~1 h and tests the more important question.

## Experimental design
A/B on the **same 500 dev rows** so the comparison is clean, then scale the winner.

| Bar | Token F1 |
|---|---|
| TRAIN-01 beam-4 baseline | 0.2051 |
| Constant string (on LB at 0.57849) | **0.2669** |
| Frequency-only constant | **0.3519** |

In [ ]:
# ══ 1 — HARDWARE GATE + PINNED LIBS ═════════════════════════════════════════
import torch
n = torch.cuda.device_count()
assert n > 0, "No GPU. Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0)
print(f"torch {torch.__version__} | {[torch.cuda.get_device_name(i) for i in range(n)]} | sm_{cap[0]}{cap[1]}")
assert cap[0] >= 7, f"WRONG ACCELERATOR sm_{cap[0]}{cap[1]} — use GPU T4 x2"
print("✅ hardware OK")

In [ ]:
# ══ 2 — PIN transformers (trap #00: Kaggle ships 5.0.0, which breaks T5) ════
!pip install -q --upgrade "transformers==4.57.3" git+https://github.com/csebuetnlp/normalizer
import transformers
assert transformers.__version__ == "4.57.3", transformers.__version__
from normalizer import normalize
print("transformers:", transformers.__version__)
print("normalizer OK:", normalize("হেলো,   নাসেনিয়া ডকে  আপনাকে স্বাগতম।"))

In [ ]:
# ══ 3 — locate code, data, and the CHECKPOINT ═══════════════════════════════
import glob, os, shutil, sys
print("/kaggle/input:", os.listdir("/kaggle/input"))

hits = glob.glob("/kaggle/input/**/04_decode.py", recursive=True)
assert hits, "Attach the `nascenia-code` dataset"
CODE = os.path.dirname(hits[0])

raw = glob.glob("/kaggle/input/**/train.csv", recursive=True)
assert raw, "Attach the Nascenia AI Hackathon competition"
RAW = os.path.dirname(raw[0])

# the checkpoint: any attached dir with config.json beside model weights,
# excluding the code dataset itself
cands = []
for cfg in glob.glob("/kaggle/input/**/config.json", recursive=True):
    d = os.path.dirname(cfg)
    if glob.glob(f"{d}/*.safetensors") or glob.glob(f"{d}/*.bin"):
        cands.append(d)
assert cands, (
    "NO CHECKPOINT FOUND.\n"
    "On the training notebook: File -> Save Version -> QUICK SAVE (not Save & Run All,\n"
    "which re-runs from scratch and discards the trained weights). Then attach it here:\n"
    "  Add Input -> Notebooks -> nascenia-banglat5-v2\n"
    f"searched: {os.listdir('/kaggle/input')}"
)
CKPT = sorted(cands, key=lambda p: ('best' not in p, len(p)))[0]

os.makedirs("/kaggle/working/code", exist_ok=True)
for f in glob.glob(f"{CODE}/*.py"):
    shutil.copy(f, "/kaggle/working/code/")
sys.path.insert(0, "/kaggle/working/code")
print("CODE:", CODE, "\nRAW :", RAW, "\nCKPT:", CKPT)
print("  contents:", sorted(os.listdir(CKPT))[:8])
if len(cands) > 1:
    print(f"  {len(cands)} checkpoints attached, chose the one above. all: {cands}")

In [ ]:
# ══ 4 — rebuild the identical frozen dev split ══════════════════════════════
# seed 42 / dev-size 5000 must match every other run or dev scores are incomparable.
!cd /kaggle/working/code && python 01_prep.py --raw "{RAW}" --out /kaggle/working/processed --seed 42 --dev-size 5000

## A — beam-4 baseline on 500 dev rows
Reproduces TRAIN-01's decoding so the MBR comparison is like-for-like.
Expect Token F1 ≈ **0.205**. `--no-bertscore` because BERTScore is mis-calibrated
locally and near-constant anyway — judge on Token F1 / ROUGE-L.

In [ ]:
!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --mode beam --limit 500 \
    --num-beams 4 --min-new-tokens 80 --no-bertscore \
    --record /kaggle/working/beam500.json

## B — MBR on the *same* 500 rows
16 sampled candidates, utility = `0.3·TokenF1 + 0.2·ROUGE-L` against the other samples.

**This is the experiment.** If MBR ≫ beam, the diagnosis (specifics are penalised,
consensus wins) is right and everything downstream follows from it.

In [ ]:
!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --mode mbr -n 16 --limit 500 \
    --temperature 0.8 --top-p 0.95 --min-new-tokens 80 --no-bertscore \
    --record /kaggle/working/mbr16_500.json

## C — more candidates, if B improved
MBR quality rises with N. Only worth the time if B already beat beam.

In [ ]:
!cd /kaggle/working/code && python 04_decode.py \
    --ckpt "{CKPT}" --split dev --mode mbr -n 32 --limit 500 \
    --temperature 0.8 --top-p 0.95 --min-new-tokens 80 --no-bertscore \
    --record /kaggle/working/mbr32_500.json

In [ ]:
# ══ 5 — comparison table ════════════════════════════════════════════════════
import json, glob
rows = []
for f in sorted(glob.glob("/kaggle/working/*.json")):
    d = json.load(open(f))
    dv = d.get("dev", {})
    rows.append((os.path.basename(f), d.get("mode"), d.get("n_candidates"),
                 dv.get("token_f1"), dv.get("rouge_l"), dv.get("mean_pred_tokens")))

print(f"{'run':22s} {'mode':6s} {'N':>4s} {'TokenF1':>9s} {'ROUGE-L':>9s} {'lexical':>9s} {'tokens':>7s}")
for name, mode, n, f1, rl, tk in rows:
    if f1 is None:
        continue
    lex = 0.3 * f1 + 0.2 * rl
    print(f"{name:22s} {str(mode):6s} {str(n):>4s} {f1:9.4f} {rl:9.4f} {lex:9.4f} {tk:7.1f}")

print("\nbars —  beam baseline 0.2051 | constant 0.2669 | frequency-only 0.3519")
print("predicted LB = 0.4672 + 0.3*TokenF1 + 0.2*ROUGE-L")

---
## Reading the result

| MBR Token F1 vs beam's 0.2051 | Meaning | Next |
|---|---|---|
| **> 0.2669** | beats the constant — MBR is the mechanism | scale to full 5k, pool seed 1337, submit |
| 0.22 – 0.2669 | MBR helps but not enough alone | add length tuning + second seed |
| ≈ 0.205 (no change) | consensus selection is not the lever | the model is undertrained — retrain at `lr 3e-4 / warmup 200` |

Record whichever happens in `LOCAL_EXPERIMENTS.md`, including a negative result.